In [30]:
# install dependencies
!pip install -q gradio google-generativeai PyMuPDF

# imports
import os
import gradio as gr
import fitz
import google.generativeai as genai

from kaggle_secrets import UserSecretsClient
secrets = UserSecretsClient()
api_key = secrets.get_secret("main_key")  

genai.configure(api_key=api_key)
model = genai.GenerativeModel("models/gemma-3-4b-it")

def score_depression(text):
    red_flags = [
        "hopeless", "empty", "worthless", "suicide", "tired of living",
        "no purpose", "can't go on", "lost", "numb", "want to die"
    ]
    score = sum(word in text.lower() for word in red_flags)
    return min(score, 3)

session = {"score": 0, "log": []}
user_sex = None
user_age = None

def extract_text_from_pdf(filepath):
    try:
        doc = fitz.open(filepath)
        content = ""
        for page in doc:
            content += page.get_text()
        doc.close()
        return content
    except Exception as e:
        return ""

def generate_reply(user_msg, previous_conver):
    history_context = "\n".join([f"User: {user}\nPsychologist: {bot}" for user, bot in previous_conver[-3:]])
    prompt = f"""
You are a compassionate psychologist. Build trust, ask reflective questions, and help the user explore their feelings.

Previous context:
{history_context}

Current user input:
"{user_msg}"

Write a supportive, thoughtful response .
"""
    return model.generate_content(prompt).text.strip()

def analyze_uploaded_file(file, chatbot):
    if file is None:
        chatbot.append({"role": "assistant", "content": "Please upload a file first."})
        return chatbot

    try:
        filename = file.name if hasattr(file, 'name') else file

        if filename.endswith(".pdf"):
            doc = fitz.open(filename)
            text_data = ""
            for page in doc:
                text_data += page.get_text()
            doc.close()
        elif filename.endswith(".txt"):
            with open(filename, "r", encoding="utf-8") as f:
                text_data = f.read()
        else:
            chatbot.append({"role": "assistant", "content": "Unsupported file type."})
            return chatbot

        if len(text_data.strip()) == 0:
            chatbot.append({"role": "assistant", "content": "Uploaded document seems empty or not readable."})
            return chatbot

        doc_score = score_depression(text_data)
        session["score"] += doc_score
        feedback = generate_reply(text_data[:500], [])

        chatbot.append({"role": "assistant", "content": f"📝 Analyzed your uploaded document.\n\n{feedback}"})
        return chatbot  

    except Exception as e:
        chatbot.append({"role": "assistant", "content": f"Error analyzing file: {str(e)}"})
        return chatbot



def chat(user_message, chat_history):
    message_score = score_depression(user_message)
    session["score"] += message_score
    reply = generate_reply(user_message, session["log"])
    session["log"].append((user_message, reply))
    chat_history.append({"role": "user", "content": user_message})
    chat_history.append({"role": "assistant", "content": reply})
    return chat_history, ""

def summarize_conversation():
    full_convo = "\n".join([f"User: {user}\nPsychologist: {bot}" for user, bot in session["log"]])
    total_score = session["score"]
    
    if total_score <= 3:
        risk = "Low Risk"
    elif total_score <= 6:
        risk = "Moderate Risk"
    else:
        risk = "High Risk"

    prompt = f"""
You are a clinical psychologist reviewing the conversation.

Transcript:
{full_convo}

The user's depression risk score is {total_score} ({risk}).

Write:
1. A summary of their emotional state
2. If signs of depression are present
3. 3 helpful mental health tips
4. Encouragement to seek professional help
"""
    summary_text = model.generate_content(prompt).text.strip()
    return f"Depression Risk: {risk}\n\n{summary_text}"

def start_session():
    return gr.update(visible=True), gr.update(visible=True), gr.update(visible=True)

def submit_info(sex, age):
    global user_sex, user_age
    user_sex = sex
    user_age = age
    return (
        gr.update(visible=True),
        gr.update(visible=True),
        gr.update(visible=True),
        gr.update(visible=True),
        gr.update(visible=True),
        gr.update(visible=True)
    )

def finish_session():
    summary = summarize_conversation()
    return summary

with gr.Blocks() as app:
    start_button = gr.Button("Start")
    sex_input = gr.Radio(["Male", "Female", "Other"], label="Sex", visible=False)
    age_input = gr.Number(label="Age", precision=0, visible=False)
    submit_info_button = gr.Button("Submit Info", visible=False)

    chatbot = gr.Chatbot(label="Mental Health Assistant", visible=False, type="messages")
    user_message = gr.Textbox(label="Your message...", placeholder="Share how you're feeling", visible=False)
    finish_btn = gr.Button("Finish and Get Full Report", visible=False)
    output_summary = gr.Textbox(label="Final Psychologist Report", lines=10, visible=False)
    upload_file = gr.File(file_types=[".pdf", ".txt"], label="(Optional) Upload a Document for deeper analysis", visible=False)
    upload_btn = gr.Button("Analyze Uploaded File", visible=False)

    start_button.click(start_session, outputs=[sex_input, age_input, submit_info_button])
    submit_info_button.click(
        submit_info, inputs=[sex_input, age_input],
        outputs=[chatbot, user_message, finish_btn, output_summary, upload_file, upload_btn]
    )

    user_message.submit(chat, [user_message, chatbot], [chatbot, user_message])
    finish_btn.click(finish_session, outputs=[output_summary])
    upload_btn.click(analyze_uploaded_file, [upload_file, chatbot], [chatbot])

app.launch()


* Running on local URL:  http://127.0.0.1:7880
It looks like you are running Gradio on a hosted a Jupyter notebook. For the Gradio app to work, sharing must be enabled. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

* Running on public URL: https://d3dae08579e4d47c6e.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
